In [2]:
import pandas as pd
import numpy as np
from patsy.contrasts import Treatment

import pandas as pd
# import statsmodels.api as sm

import statsmodels.formula.api as smf

# from sklearn.preprocessing import LabelEncoder

from statsmodels.stats.anova import anova_lm


import plotly.express as px

Method info:

https://medium.com/swlh/interpreting-linear-regression-through-statsmodels-summary-4796d359035a

In [6]:
#load infectivity data and log10 transform
infdf = pd.read_csv('infectivity_210425.tsv', sep='\t', index_col = 0)
infdf_log10 = infdf.applymap(np.log10)
infdf_log10

C:\Users\slytras\AppData\Local\Temp\ipykernel_29124\1091852367.py:3: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  infdf_log10 = infdf.applymap(np.log10)


,RhGB01,BM48-31,BtKY72,RsSHC014,YN2020H,BtRs-BetaCoV/YN2020-Q325,Rs7327,RsYN2016A,WIV1,WIV16,...,MP789 (MpCoV-GD),cDNA16,cDNA8,Guangdong/1/2019,Rp22DB159,CX1,SARS-CoV-2,BANAL-20-52,BANAL-20-236,Ra22DB163
01_QMQ39222,0.683616,0.368671,3.628956,3.646933,2.650310,3.092896,3.003230,3.058371,4.734033,4.678513,...,4.078656,3.537222,4.307903,4.819834,3.866591,3.355814,3.109106,4.003961,3.949026,1.299034
02_HDA,1.685716,0.000000,4.156571,3.913596,2.900248,3.050653,2.909316,2.732306,4.803170,4.703441,...,4.082908,3.512619,4.446472,4.868950,3.781587,3.789164,3.034989,4.138240,4.094987,2.585621
03_LHD,3.177940,0.262684,4.274139,3.946759,3.133058,3.391257,3.162197,2.949331,4.769975,4.739583,...,4.131106,3.572691,4.734178,4.978869,3.984757,3.996901,3.226256,4.228704,4.173946,3.019065
04_LHDH,3.054980,0.518578,4.209918,3.798360,2.819975,3.290436,3.183628,2.995909,4.604968,4.592366,...,4.084395,3.555541,4.666357,4.968319,4.064192,3.956261,3.236636,4.065306,4.092442,2.998251
05_LHDAH,3.318207,0.514482,4.277500,3.781857,2.858013,3.216496,3.149583,2.861980,4.604018,4.595188,...,4.025857,3.486585,4.635372,4.897716,4.007920,3.971351,3.261932,4.072925,4.077667,2.989598
06_GHDAH,2.176924,0.486350,4.040585,3.698457,2.732936,2.992713,2.800733,2.556346,4.496877,4.395834,...,3.993966,3.309431,4.377009,4.690558,3.765946,3.676860,3.082138,3.964163,3.921900,2.383783
07_LHDA,3.147289,0.710226,4.200885,3.914038,2.981433,3.276454,3.096343,2.953426,4.591689,4.606353,...,4.030580,3.501110,4.657598,4.886598,3.995683,3.975266,3.263438,4.089623,4.094610,2.974214
09_HD,1.558058,0.484634,4.068865,3.893386,3.042633,3.140514,2.781856,2.697889,4.775480,4.576089,...,4.190640,3.617970,4.480197,4.898600,3.941606,3.863409,3.118153,4.058294,4.053253,2.622869
10_HDH,1.830809,0.510037,4.100481,3.819668,2.942942,2.831726,2.846312,2.628220,4.698235,4.539473,...,4.026535,3.393583,4.397741,4.803822,3.759094,3.716571,3.173542,3.952033,3.962590,2.358524
12_HDAH,1.864311,0.786067,3.804039,3.551364,2.575934,2.700713,2.711951,2.515818,4.416672,4.313460,...,3.916480,3.422828,4.228907,4.677140,3.606882,3.580450,3.071088,3.763627,3.731800,2.226817


In [7]:
#load ace2 residue table
ace2df = pd.read_csv('ACE2_table_140425.tsv', sep='\t', index_col = 0)
ace2df

,24,34,38,49,83,185,224,300,603
01_QMQ39222,R,R,E,E,Y,V,K,Q,Y
02_HDA,R,H,D,E,Y,A,K,Q,Y
03_LHD,L,H,D,E,Y,V,K,Q,Y
04_LHDH,L,H,D,E,Y,V,K,Q,H
05_LHDAH,L,H,D,E,Y,A,K,Q,H
06_GHDAH,G,H,D,E,Y,A,K,Q,H
07_LHDA,L,H,D,E,Y,A,K,Q,Y
09_HD,R,H,D,E,Y,V,K,Q,Y
10_HDH,R,H,D,E,Y,V,K,Q,H
12_HDAH,R,H,D,E,Y,A,K,Q,H


In [8]:
#concatenate together
concatdf = pd.concat([ace2df, infdf_log10], axis=1)
concatdf

,24,34,38,49,83,185,224,300,603,RhGB01,...,MP789 (MpCoV-GD),cDNA16,cDNA8,Guangdong/1/2019,Rp22DB159,CX1,SARS-CoV-2,BANAL-20-52,BANAL-20-236,Ra22DB163
01_QMQ39222,R,R,E,E,Y,V,K,Q,Y,0.683616,...,4.078656,3.537222,4.307903,4.819834,3.866591,3.355814,3.109106,4.003961,3.949026,1.299034
02_HDA,R,H,D,E,Y,A,K,Q,Y,1.685716,...,4.082908,3.512619,4.446472,4.868950,3.781587,3.789164,3.034989,4.138240,4.094987,2.585621
03_LHD,L,H,D,E,Y,V,K,Q,Y,3.177940,...,4.131106,3.572691,4.734178,4.978869,3.984757,3.996901,3.226256,4.228704,4.173946,3.019065
04_LHDH,L,H,D,E,Y,V,K,Q,H,3.054980,...,4.084395,3.555541,4.666357,4.968319,4.064192,3.956261,3.236636,4.065306,4.092442,2.998251
05_LHDAH,L,H,D,E,Y,A,K,Q,H,3.318207,...,4.025857,3.486585,4.635372,4.897716,4.007920,3.971351,3.261932,4.072925,4.077667,2.989598
06_GHDAH,G,H,D,E,Y,A,K,Q,H,2.176924,...,3.993966,3.309431,4.377009,4.690558,3.765946,3.676860,3.082138,3.964163,3.921900,2.383783
07_LHDA,L,H,D,E,Y,A,K,Q,Y,3.147289,...,4.030580,3.501110,4.657598,4.886598,3.995683,3.975266,3.263438,4.089623,4.094610,2.974214
09_HD,R,H,D,E,Y,V,K,Q,Y,1.558058,...,4.190640,3.617970,4.480197,4.898600,3.941606,3.863409,3.118153,4.058294,4.053253,2.622869
10_HDH,R,H,D,E,Y,V,K,Q,H,1.830809,...,4.026535,3.393583,4.397741,4.803822,3.759094,3.716571,3.173542,3.952033,3.962590,2.358524
12_HDAH,R,H,D,E,Y,A,K,Q,H,1.864311,...,3.916480,3.422828,4.228907,4.677140,3.606882,3.580450,3.071088,3.763627,3.731800,2.226817


In [9]:
#melt the virus columns and change the ACE2 site numbers to alphabetical
fullmelt = pd.melt(concatdf, id_vars=concatdf.columns[:9], value_vars=concatdf.columns[-36:], var_name='virus')
fullmelt.columns = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'virus', 'value' ]
fullmelt

,A,B,C,D,E,F,G,H,I,virus,value
0,R,R,E,E,Y,V,K,Q,Y,RhGB01,0.683616
1,R,H,D,E,Y,A,K,Q,Y,RhGB01,1.685716
2,L,H,D,E,Y,V,K,Q,Y,RhGB01,3.177940
3,L,H,D,E,Y,V,K,Q,H,RhGB01,3.054980
4,L,H,D,E,Y,A,K,Q,H,RhGB01,3.318207
...,...,...,...,...,...,...,...,...,...,...,...
607,R,R,E,E,H,V,K,Q,Y,Ra22DB163,1.642723
608,R,R,E,E,Y,V,K,Q,S,Ra22DB163,1.304504
609,R,R,E,D,H,V,K,Q,Y,Ra22DB163,1.678214
610,R,R,E,E,Y,V,K,H,Y,Ra22DB163,1.380469


In [10]:
#fix reference for sites of interest by adding alphabetical prefix
refdic = {'A':['R','A'], 'B':['R', 'A'], 'C':['E','A'], 'D':['E','A'], 'E':['Y', 'A']}
for site in list(refdic):
    fullmelt[site] = fullmelt[site].replace(refdic[site][0], refdic[site][1])
    
fullmelt

,A,B,C,D,E,F,G,H,I,virus,value
0,A,A,A,A,A,V,K,Q,Y,RhGB01,0.683616
1,A,H,D,A,A,A,K,Q,Y,RhGB01,1.685716
2,L,H,D,A,A,V,K,Q,Y,RhGB01,3.177940
3,L,H,D,A,A,V,K,Q,H,RhGB01,3.054980
4,L,H,D,A,A,A,K,Q,H,RhGB01,3.318207
...,...,...,...,...,...,...,...,...,...,...,...
607,A,A,A,A,H,V,K,Q,Y,Ra22DB163,1.642723
608,A,A,A,A,A,V,K,Q,S,Ra22DB163,1.304504
609,A,A,A,D,H,V,K,Q,Y,Ra22DB163,1.678214
610,A,A,A,A,A,V,K,H,Y,Ra22DB163,1.380469


In [11]:
#do the regression with sites as expalantory variables for each virus separately

olsrez = {}

for v in set(fullmelt.virus):
    
    mod = smf.ols(formula="value ~ A + B + C + D + E + F + G + H + I",data=fullmelt[fullmelt.virus == v])
    
    res = mod.fit()

    olsrez.update({v:res})


In [12]:
olsrez['RaTG13'].summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                  value   R-squared:                       0.993
Model:                            OLS   Adj. R-squared:                  0.980
Method:                 Least Squares   F-statistic:                     79.71
Date:                Thu, 10 Sep 2026   Prob (F-statistic):           1.43e-05
Time:                        23:50:01   Log-Likelihood:                 13.943
No. Observations:                  17   AIC:                            -5.886
Df Residuals:                       6   BIC:                             3.280
Df Model:                          10                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.0442      0.254      0.174      0.867      -0.576       0.665
A[T.G]        -0.2710      0.220     -1.234      0.263      -0.808       0.267
A[T.L]         0.8808      0.127      6.945      0.000       0.570       1.191
B[T.H]         0.9819      0.097     10.137      0.000       0.745       1.219
C[T.D]         0.9819      0.097     10.137      0.000       0.745       1.219
D[T.D]        -0.4542      0.146     -3.102      0.021      -0.813      -0.096
E[T.H]         0.4247      0.179      2.368      0.056      -0.014       0.864
F[T.V]         0.1758      0.127      1.387      0.215      -0.134       0.486
G[T.T]        -0.2300      0.232     -0.993      0.359      -0.797       0.337
H[T.Q]         0.4846      0.232      2.093      0.081      -0.082       1.051
I[T.S]         0.0752      0.220      0.342      0.744      -0.462       0.613
I[T.Y]         0.1860      0.127      1.467      0.193      -0.124       0.496
==============================================================================
Omnibus:                        2.194   Durbin-Watson:                   2.106
Prob(Omnibus):                  0.334   Jarque-Bera (JB):                0.966
Skew:                          -0.574   Prob(JB):                        0.617
Kurtosis:                       3.217   Cond. No.                     5.22e+16
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The smallest eigenvalue is 2.18e-32. This might indicate that there are
strong multicollinearity problems or that the design matrix is singular.
"""

In [13]:
#summarise overall OLS p-value and run a Bonferroni correction for all viruses
pvaldf = pd.DataFrame([[v,olsrez[v].f_pvalue] for v in list(olsrez)])
pvaldf.columns = ['virus', 'pval']
pvaldf.sort_values(by='pval', inplace=True)
pvaldf['Bonferroni'] = ['sig' if p < 0.05/len(pvaldf) else 'nonsig' for p in list(pvaldf.pval)]
pvaldf.to_csv('infectivity_210425_ols_all.csv', index=False)
pvaldf

,virus,pval,Bonferroni
26,Ra22QT77,6.308963e-07,sig
1,CX1,3.582062e-06,sig
16,GX-P5E,7.230664e-06,sig
0,RaTG13,1.427171e-05,sig
4,GX-P1E (MpCoV-GX),3.368961e-05,sig
22,cDNA8,2.364899e-04,sig
33,Ra22DB163,5.374126e-04,sig
9,RhGB01,6.878209e-04,sig
29,RShSTT200,8.497571e-04,sig
27,Ra7909,1.162774e-03,sig


In [12]:
#export all individual virus results

#also replace sites alphabetical with actual positions:
alph2num = {'A[':'24[', 'B[':'34[', 'C[':'38[', 'D[':'49[', 'E[':'83[','F[':'185[','G[':'224[','H[':'300[', 'I[':'603['}
for v in list(pvaldf.virus):
    with open('infectivity_210425_ols_%s.txt'%v.replace(' ', '').replace('/', ''), 'w') as out:
        for i in range(3):
            outstr = str(olsrez[v].summary().tables[i])
            if i==1:
                for an in list(alph2num):
                    outstr = outstr.replace(an, alph2num[an])
            out.write(outstr)
    

<br>
<br>
<br>

## Model robustness testing:

Ensure that an additive model is the best fit for the analysis by testing whether adding pairwise epistatic interactions between tested residues improves the model fit. 



<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                  value   R-squared:                       0.997
Model:                            OLS   Adj. R-squared:                  0.993
Method:                 Least Squares   F-statistic:                     227.6
Date:                Thu, 10 Sep 2026   Prob (F-statistic):           6.31e-07
Time:                        23:51:46   Log-Likelihood:                 29.842
No. Observations:                  17   AIC:                            -37.68
Df Residuals:                       6   BIC:                            -28.52
Df Model:                          10                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      4.0534      0.100     40.716      0.000       3.810       4.297
A[T.G]        -2.5540      0.086    -29.624      0.000      -2.765      -2.343
A[T.L]        -1.2015      0.050    -24.138      0.000      -1.323      -1.080
B[T.H]        -0.1750      0.038     -4.603      0.004      -0.268      -0.082
C[T.D]        -0.1750      0.038     -4.603      0.004      -0.268      -0.082
D[T.D]        -0.0612      0.057     -1.065      0.328      -0.202       0.079
E[T.H]        -0.7577      0.070    -10.764      0.000      -0.930      -0.585
F[T.V]         0.0755      0.050      1.517      0.180      -0.046       0.197
G[T.T]        -0.0565      0.091     -0.622      0.557      -0.279       0.166
H[T.Q]         0.0460      0.091      0.506      0.631      -0.176       0.268
I[T.S]        -0.0285      0.086     -0.330      0.752      -0.239       0.182
I[T.Y]        -0.0055      0.050     -0.110      0.916      -0.127       0.116
==============================================================================
Omnibus:                        0.336   Durbin-Watson:                   1.556
Prob(Omnibus):                  0.845   Jarque-Bera (JB):                0.177
Skew:                          -0.218   Prob(JB):                        0.915
Kurtosis:                       2.756   Cond. No.                     5.22e+16
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The smallest eigenvalue is 2.18e-32. This might indicate that there are
strong multicollinearity problems or that the design matrix is singular.
"""

In [15]:
#then fit model with all pairwise interactions
pairwise_formula = 'value ~ (A + B + C + D + E + F + G + H + I)**2'
pair_model = smf.ols(pairwise_formula,data=fullmelt[fullmelt.virus == 'Ra22QT77']).fit()
pair_model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                  value   R-squared:                       0.999
Model:                            OLS   Adj. R-squared:                  0.983
Method:                 Least Squares   F-statistic:                     61.27
Date:                Thu, 10 Sep 2026   Prob (F-statistic):              0.100
Time:                        23:53:05   Log-Likelihood:                 37.349
No. Observations:                  17   AIC:                            -42.70
Df Residuals:                       1   BIC:                            -29.37
Df Model:                          15                                         
Covariance Type:            nonrobust                                         
=================================================================================
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
Intercept         1.8823      0.055     34.058      0.019       1.180       2.585
A[T.G]           -0.6398      0.038    -16.857      0.038      -1.122      -0.158
A[T.L]           -0.2934      0.034     -8.643      0.073      -0.725       0.138
B[T.H]            0.2615      0.018     14.592      0.044       0.034       0.489
C[T.D]            0.2615      0.018     14.592      0.044       0.034       0.489
D[T.D]           -0.0290      0.032     -0.916      0.528      -0.432       0.373
E[T.H]           -0.2118      0.039     -5.403      0.117      -0.710       0.286
F[T.V]            1.1433      0.070     16.257      0.039       0.250       2.037
G[T.T]           -0.0264      0.039     -0.673      0.623      -0.524       0.472
H[T.Q]            0.5650      0.060      9.408      0.067      -0.198       1.328
I[T.S]            0.2404      0.046      5.243      0.120      -0.342       0.823
I[T.Y]            1.0214      0.080     12.761      0.050       0.004       2.038
A[T.G]:B[T.H]    -0.6398      0.038    -16.857      0.038      -1.122      -0.158
A[T.L]:B[T.H]    -0.2934      0.034     -8.643      0.073      -0.725       0.138
A[T.G]:C[T.D]    -0.6398      0.038    -16.857      0.038      -1.122      -0.158
A[T.L]:C[T.D]    -0.2934      0.034     -8.643      0.073      -0.725       0.138
A[T.G]:D[T.D]          0          0        nan        nan           0           0
A[T.L]:D[T.D]          0          0        nan        nan           0           0
A[T.G]:E[T.H]          0          0        nan        nan           0           0
A[T.L]:E[T.H]          0          0        nan        nan           0           0
A[T.G]:F[T.V]          0          0        nan        nan           0           0
A[T.L]:F[T.V]     0.0463      0.157      0.295      0.817      -1.946       2.039
A[T.G]:G[T.T]          0          0        nan        nan           0           0
A[T.L]:G[T.T]          0          0        nan        nan           0           0
A[T.G]:H[T.Q]    -0.6398      0.038    -16.857      0.038      -1.122      -0.158
A[T.L]:H[T.Q]    -0.2934      0.034     -8.643      0.073      -0.725       0.138
A[T.G]:I[T.S]          0          0        nan        nan           0           0
A[T.L]:I[T.S]          0          0        nan        nan           0           0
A[T.G]:I[T.Y]          0          0        nan        nan           0           0
A[T.L]:I[T.Y]    -0.1021      0.157     -0.651      0.633      -2.094       1.890
B[T.H]:C[T.D]     0.2615      0.018     14.592      0.044       0.034       0.489
B[T.H]:D[T.D]          0          0        nan        nan           0           0
B[T.H]:E[T.H]          0          0        nan        nan           0           0
B[T.H]:F[T.V]    -0.4776      0.038    -12.465      0.051      -0.964       0.009
B[T.H]:G[T.T]          0          0        nan   

In [24]:
addepi_rez = {}

#for each virus:
for v in list(pvaldf.virus):
    #fit additive model (as above):
    add_model = smf.ols(formula="value ~ A + B + C + D + E + F + G + H + I",data=fullmelt[fullmelt.virus == v]).fit()

    #then fit model with all pairwise interactions
    pairwise_formula = 'value ~ (A + B + C + D + E + F + G + H + I)**2'
    pair_model = smf.ols(pairwise_formula,data=fullmelt[fullmelt.virus == v]).fit()

    #finally perform an ANOVA nested model comparison test between the two models:
    anova_results = anova_lm(add_model, pair_model)

    addepi_rez.update({v:anova_results.iloc[1]['Pr(>F)']})

In [26]:
#summarise overall nested model ANOVA p-value and run a Bonferroni correction for all viruses
addepi_pvaldf = pd.DataFrame([[v,addepi_rez[v]] for v in list(addepi_rez)])
addepi_pvaldf.columns = ['virus', 'pval']
addepi_pvaldf.sort_values(by='pval', inplace=True)
addepi_pvaldf['Bonferroni'] = ['sig' if p < 0.05/len(pvaldf) else 'nonsig' for p in list(addepi_pvaldf.pval)]
addepi_pvaldf

,virus,pval,Bonferroni
18,RsYN2016A,0.021764,nonsig
2,GX-P5E,0.030735,nonsig
4,GX-P1E (MpCoV-GX),0.031222,nonsig
11,SARS-CoV-2,0.033087,nonsig
13,Rc-os20,0.059153,nonsig
28,Rc-o319,0.141986,nonsig
6,Ra22DB163,0.166885,nonsig
19,Rp22DB159,0.188627,nonsig
27,MP789 (MpCoV-GD),0.210419,nonsig
21,BtRs-BetaCoV/YN2020-Q325,0.230985,nonsig


**Conclusion:** The model fit does not significantly improve by adding epistatic interactions for any of the viruses tested.